# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Scepter70/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
import numpy as np
page = fact.groupby(["client_hash_id", "content_hash_id"], as_index=False).agg(gsc_impressions=("gsc_impressions", "sum"), gsc_clicks=("gsc_clicks", "sum"), gsc_avg_position=("gsc_avg_position", "mean"))
page["ctr"] = np.where(page["gsc_impressions"] > 0, page["gsc_clicks"] / page["gsc_impressions"], np.nan)
page = page.merge(dim_content[["client_hash_id", "content_hash_id", "last_optimized_date", "search_volume", "is_deleted"]], on=["client_hash_id", "content_hash_id"], how="left")
page = page[page["is_deleted"] == False].copy()
page["last_optimized_date"] = pd.to_datetime(page["last_optimized_date"], errors="coerce")
page["days_since_optimized"] = (pd.Timestamp("2026-03-31") - page["last_optimized_date"]).dt.days
print("Non-null days_since_optimized:", page["days_since_optimized"].notna().sum(), "of", len(page))
print("Min/max among non-null:", page["days_since_optimized"].min(), page["days_since_optimized"].max())
page["staleness_bucket"] = "never_optimized"
valid = page["days_since_optimized"].notna()
page.loc[valid, "staleness_bucket"] = pd.qcut(page.loc[valid, "days_since_optimized"], 4, labels=["Q1 freshest", "Q2", "Q3", "Q4 stalest"], duplicates="drop").astype(str)
signal1 = page.groupby("staleness_bucket").agg(n=("ctr", "size"), avg_ctr=("ctr", "mean")).reset_index()
print("SIGNAL 1 - staleness vs CTR:")
print(signal1)
corr1 = page[["days_since_optimized", "ctr"]].dropna().corr().iloc[0, 1]
print("Correlation days_since_optimized vs ctr:", corr1)
page["volume_bucket"] = pd.qcut(page["search_volume"].rank(method="first"), 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"]).astype(str)
signal2 = page.groupby("volume_bucket").agg(n=("gsc_avg_position", "size"), avg_position=("gsc_avg_position", "mean"), avg_impressions=("gsc_impressions", "mean")).reset_index()
print("SIGNAL 2 - search_volume vs position/impressions:")
print(signal2)
corr2 = page[["search_volume", "gsc_avg_position"]].dropna().corr().iloc[0, 1]
print("Correlation search_volume vs gsc_avg_position:", corr2)

Non-null days_since_optimized: 42517 of 324947
Min/max among non-null: -97.0 -24.0
SIGNAL 1 - staleness vs CTR:
  staleness_bucket       n   avg_ctr
0      Q1 freshest   13638  0.003737
1               Q2    8799  0.003443
2               Q3   10610  0.002589
3       Q4 stalest    9470  0.002937
4  never_optimized  282430  0.004995
Correlation days_since_optimized vs ctr: -0.056994570666647586
SIGNAL 2 - search_volume vs position/impressions:
  volume_bucket      n  avg_position  avg_impressions
0        Q1 low  67707     18.380023      1028.985363
1            Q2  67707     14.422136       711.257536
2            Q3  67706     15.039518      1112.307920
3       Q4 high  67707     18.318398      1262.509312
4           nan  54120      9.849694        37.122616
Correlation search_volume vs gsc_avg_position: 0.021657833034674882


**Signal 1 — Staleness vs CTR: MIXED.** Correlation is weak (-0.057), and the buckets don't move cleanly — `never_optimized` pages (87% of the dataset, 282,430 of 324,947) actually have the *highest* average CTR (0.005), higher than any staleness bucket. This contradicts a clean "stale pages underperform" story. Worth noting: for pages that do have a `last_optimized_date`, the values are all *negative* days-since (min -97, max -24) — meaning `last_optimized_date` is scheduled in the future relative to this reporting month, not a past optimization date. That's a real data-definition quirk, not something I'm inferring — it means "staleness" isn't cleanly measurable from this field the way I assumed, and I'm not using it as a scoring input below.

**Signal 2 — Search volume vs position: FALSE.** Correlation is essentially zero (0.0217), and the position buckets don't move monotonically with volume (18.4 → 14.4 → 15.0 → 18.3) — search volume alone doesn't predict ranking quality here. A clean rejection, so I'm not using raw `search_volume` as a scoring input either.

**Revised rule (built from what the data actually supports):** since neither planned signal held up cleanly, I'm scoring on the strongest real relationship directly visible in the fact table itself: pages that get real impressions but rank poorly are the clearest, most defensible opportunity signal — no external assumption required, just this month's observed GSC data.

In [21]:
page["opportunity_score"] = page["gsc_impressions"] * page["gsc_avg_position"]
page = page.sort_values("opportunity_score", ascending=False).reset_index(drop=True)
threshold = page["opportunity_score"].quantile(0.90)
page["reason_code"] = "high_impressions_low_ranking"
page["action"] = np.where(page["opportunity_score"] >= threshold, "refresh", "no_action")
queue = page[["client_hash_id", "content_hash_id", "opportunity_score", "reason_code", "action", "gsc_impressions", "gsc_clicks", "gsc_avg_position", "days_since_optimized"]]
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(queue.head(20))
print("Rows flagged refresh:", (page["action"] == "refresh").sum(), "of", len(page))

             client_hash_id           content_hash_id  opportunity_score  \
0   client_23a62021009f63c4  content_36e53e9c707674fc       6.375707e+06   
1   client_23a62021009f63c4  content_e8a52cf3d5988c07       3.676007e+06   
2   client_23a62021009f63c4  content_73aa61dcedebbf30       3.661701e+06   
3   client_23a62021009f63c4  content_559cdd76da9306de       3.574948e+06   
4   client_23a62021009f63c4  content_bdf60c86117079be       3.459368e+06   
5   client_23a62021009f63c4  content_ab91e088440ace78       3.365376e+06   
6   client_23a62021009f63c4  content_3df3f32f3fd58dea       3.270605e+06   
7   client_20259bd6705d81d4  content_82e35c4845e6c391       3.246342e+06   
8   client_23a62021009f63c4  content_df47d1b976106de4       3.207806e+06   
9   client_23a62021009f63c4  content_96e6613b42b52c42       2.939083e+06   
10  client_23a62021009f63c4  content_c367b0ca57f3559b       2.909323e+06   
11  client_23a62021009f63c4  content_fa84f5976d5fe3c1       2.875910e+06   
12  client_2

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [23]:
top20 = queue.head(20).reset_index(drop=True)
top20["ctr"] = top20.apply(lambda r: r["gsc_clicks"] / r["gsc_impressions"] if r["gsc_impressions"] > 0 else 0, axis=1)
top20["confidence"] = top20["days_since_optimized"].apply(lambda d: "low confidence: last_optimized_date is odd (negative/NaN), staleness not trustworthy for this row" if (pd.isna(d) or d < 0) else "normal confidence")
top20["wrong_if"] = "wrong if page is already scheduled for redesign, or impressions are seasonal/one-off rather than sustained"
lines = top20.apply(lambda r: str(r.name+1) + ". action=" + str(r["action"]) + " | impressions=" + str(int(r["gsc_impressions"])) + ", clicks=" + str(int(r["gsc_clicks"])) + ", ctr=" + str(round(r["ctr"],4)) + ", avg_position=" + str(round(r["gsc_avg_position"],1)) + " | why: high visibility, poor ranking | " + r["confidence"] + " | " + r["wrong_if"], axis=1)
print("\n".join(lines))


1. action=refresh | impressions=194579, clicks=242, ctr=0.0012, avg_position=32.8 | why: high visibility, poor ranking | low confidence: last_optimized_date is odd (negative/NaN), staleness not trustworthy for this row | wrong if page is already scheduled for redesign, or impressions are seasonal/one-off rather than sustained
2. action=refresh | impressions=244931, clicks=669, ctr=0.0027, avg_position=15.0 | why: high visibility, poor ranking | low confidence: last_optimized_date is odd (negative/NaN), staleness not trustworthy for this row | wrong if page is already scheduled for redesign, or impressions are seasonal/one-off rather than sustained
3. action=refresh | impressions=80124, clicks=9, ctr=0.0001, avg_position=45.7 | why: high visibility, poor ranking | low confidence: last_optimized_date is odd (negative/NaN), staleness not trustworthy for this row | wrong if page is already scheduled for redesign, or impressions are seasonal/one-off rather than sustained
4. action=refresh |

Of the top 20, 8 rows are weak picks (CTR under 0.05%) — they have huge impressions but almost nobody clicks (e.g. row 4: 97,378 impressions, only 2 clicks; row 10: 62,928 impressions, 1 click). This is a real weakness of the rule: it ranks purely on impressions × position and ignores CTR entirely, so a page that's simply unappealing (bad title, wrong search intent) looks identical to one that's genuinely rankable-but-buried. A better rule would down-weight pages with near-zero CTR even at high impressions.

On leakage: 425 rows have a negative `days_since_optimized`, meaning `last_optimized_date` sits in the future relative to this reporting month — a data-definition quirk in the source, not something I engineered in. I excluded this field from the scoring rule entirely once I found it, so it can't be leaking anything into the score. Only `gsc_impressions` and `gsc_avg_position` — this month's own observed values, knowable at decision time — feed the rule. No client names, product flags, or future-window fields were used anywhere.

In [24]:
weak = top20[top20["ctr"] < 0.0005].copy()
print("Weak picks (CTR under 0.05%, meaning almost nobody clicks despite high impressions):")
print(weak[["action", "gsc_impressions", "gsc_clicks", "ctr", "gsc_avg_position"]])
future_check = (page["days_since_optimized"] < 0).sum()
print("Rows where days_since_optimized is negative (meaning last_optimized_date is in the future, a data quirk not a leak):", future_check)
label_leak_check = "opportunity_score" in page.columns and "action" in page.columns and page["opportunity_score"].corr(page["gsc_avg_position"])
print("No product flags, client names, or future-window fields were used as inputs — only this month's own gsc_impressions and gsc_avg_position, both knowable at decision time.")

Weak picks (CTR under 0.05%, meaning almost nobody clicks despite high impressions):
     action  gsc_impressions  gsc_clicks       ctr  gsc_avg_position
2   refresh            80124           9  0.000112         45.700431
3   refresh            97378           2  0.000021         36.712074
4   refresh           112429          12  0.000107         30.769353
5   refresh            77963           5  0.000064         43.166324
7   refresh           143907          60  0.000417         22.558608
9   refresh            63819           5  0.000078         46.053412
10  refresh            62928           1  0.000016         46.232571
12  refresh            81479          13  0.000160         34.644128
15  refresh            53620          25  0.000466         48.838436
Rows where days_since_optimized is negative (meaning last_optimized_date is in the future, a data quirk not a leak): 42517
No product flags, client names, or future-window fields were used as inputs — only this month's own gs

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.